In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


dbutils.widgets.text("catalog", "workspace", "Catálogo")
dbutils.widgets.text("volume_schema", "default", "Schema do Volume")
dbutils.widgets.text("volume_name", "inputs", "Nome do Volume")

CATALOG = dbutils.widgets.get("catalog").strip()
VOLUME_SCHEMA = dbutils.widgets.get("volume_schema").strip()
VOLUME_NAME = dbutils.widgets.get("volume_name").strip()

BRONZE_DB = "bronze"
VOLUME_PATH = f"/Volumes/{CATALOG}/{VOLUME_SCHEMA}/{VOLUME_NAME}"

# Mapeamento: arquivo de origem -> tabela Bronze
SOURCES = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews",
}

print(f"Origem : {VOLUME_PATH}")
print(f"Destino: {CATALOG}.{BRONZE_DB}")

Origem : /Volumes/workspace/default/inputs
Destino: workspace.bronze


In [0]:

#Cria o database Bronze se nao tiver
spark.sql(
    f"""
    CREATE DATABASE IF NOT EXISTS {CATALOG}.{BRONZE_DB}
    COMMENT 'Camada Bronze: dados brutos do catálogo de filmes TMDB/IMDb, sem transformação'
    """
)


DataFrame[]

In [0]:
for file_name, table_name in SOURCES.items():
    # Lê o CSV
    df = spark.read.csv(f"{VOLUME_PATH}/{file_name}", header=True)

    # Acrescenta o timestamp do momento da inserção na Bronze
    df = df.withColumn("ingestion_datetime", F.current_timestamp())

    # Grava em formato Delta, modo Append
    df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{BRONZE_DB}.{table_name}")

    print(f"{BRONZE_DB}.{table_name} gravada")


In [0]:
df = spark.table(f"{CATALOG}.{BRONZE_DB}.tb_movies_info")
display(df.limit(10))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-07T20:45:04.641Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-07T20:45:04.641Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-07T20:45:04.641Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-07T20:45:04.641Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-07T20:45:04.641Z
284054,tt1825683,Black Panther,Black Panther,en,2018-02-13,135,Released,"King T'Challa returns home to the reclusive, technologically advanced African nation of Wakanda to serve as his country's new leader. However, T'Challa soon finds that he is challenged for the throne by factions within his own country as well as without. Using powers reserved to Wakandan kings, T'Challa assumes the Black Panther mantle to join with ex-girlfriend Nakia, the queen-mother, his princess-kid sister, members of the Dora Milaje (the Wakandan 'special forces') and an American secret agent, to prevent Wakanda from being dragged into a world war.",null,2026-10-07T20:45:04.641Z
284052,tt1211837,Doctor Strange,Doctor Strange,en,2016-10-25,115,Released,"After his career is destroyed, a brilliant but arrogant surgeon gets a new lease on life when a sorcerer takes him under her wing and trains him to defend the world against evil.",The impossibilities are endless.,2026-10-07T20:45:04.641Z
315635,tt2250912,Spider-Man: Homecoming,Spider-Man: Homecoming,en,2017-07-05,133,RELEASED,"Following the events of Captain America: Civil War, Peter Parker, with the help of his mentor Tony Stark, tries to balance his life as an ordinary high school student in Queens, New York City, with fighting crime as his superhero alter ego Spider-Man as a new threat, the Vulture, emerges.",Homework can wait. The city can't.,2026-10-07T20:45:04.641Z
283995,tt3896198,Guardians of the Galaxy Vol. 2,Guardians of the Galaxy Vol. 2,en,2017-04-19,137,Released,The Guardians must fight to keep their newfound family together as t

In [0]:
import requests
from datetime import date, datetime, timedelta

# Parâmetros de data no formato MM-DD-AAAA.
dbutils.widgets.text("data_inicio", "", "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "", "Data fim (MM-DD-AAAA)")

hoje = date.today()
data_inicio_formatada = dbutils.widgets.get("data_inicio").strip() or (hoje - timedelta(days=7)).strftime("%m-%d-%Y")
data_fim_formatada = dbutils.widgets.get("data_fim").strip() or hoje.strftime("%m-%d-%Y")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)


def mock_resposta_api(data_inicio: str, data_fim: str) -> dict:
    """Simula a resposta do endpoint: uma cotação por dia útil do período."""
    inicio = datetime.strptime(data_inicio, "%m-%d-%Y").date()
    fim = datetime.strptime(data_fim, "%m-%d-%Y").date()
    dias = [inicio + timedelta(days=i) for i in range((fim - inicio).days + 1)]
    return {
        "value": [
            {"cotacaoCompra": round(5.40 + 0.01 * i, 4), "dataHoraCotacao": f"{dia:%Y-%m-%d} 13:00:00.000"}
            for i, dia in enumerate(dias)
            if dia.weekday() < 5
        ]
    }


try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    resposta = response.json()
    print("Fonte: API do Banco Central")
except Exception as erro:
    resposta = mock_resposta_api(data_inicio_formatada, data_fim_formatada)
    print(f"Fonte: MOCK (API indisponível: {erro})")

cotacoes = resposta["value"]

print(f"Período: {data_inicio_formatada} a {data_fim_formatada}")
print(cotacoes)

Fonte: MOCK (API indisponível: HTTPSConnectionPool(host='olinda.bcb.gov.br', port=443): Max retries exceeded with url: /olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='09-30-2026'&@dataFinalCotacao='10-07-2026'&$select=dataHoraCotacao,cotacaoCompra&$format=json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7f4d97598620>: Failed to resolve 'olinda.bcb.gov.br' ([Errno -3] Temporary failure in name resolution)")))
Período: 09-30-2026 a 10-07-2026
[{'cotacaoCompra': 5.4, 'dataHoraCotacao': '2026-09-30 13:00:00.000'}, {'cotacaoCompra': 5.41, 'dataHoraCotacao': '2026-10-01 13:00:00.000'}, {'cotacaoCompra': 5.42, 'dataHoraCotacao': '2026-10-02 13:00:00.000'}, {'cotacaoCompra': 5.45, 'dataHoraCotacao': '2026-10-05 13:00:00.000'}, {'cotacaoCompra': 5.46, 'dataHoraCotacao': '2026-10-06 13:00:00.000'}, {'cotacaoCompra': 5.47, 'dataHoraCotacao': '2026-10-07 13:00:00.000'}]
